# 05A.LGBM含评分

基于 04A/04B 含评分输出训练最终 LGBM。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

import lightgbm as lgb
import joblib

In [ ]:
version_tag = "A"  # A or B

df_train = pd.read_csv(OUTPUT_DIR / f"4{version_tag}.train_valid_data_含评分_{customer_type}_{target_type}.csv").fillna(-999)
df_oot = pd.read_csv(OUTPUT_DIR / f"4{version_tag}.oot_data_含评分_{customer_type}_{target_type}.csv").fillna(-999)

col_var = [c for c in df_train.columns if c not in join_keys + ["flag", target]]

score_cols = [c for c in col_var if c.startswith("score")]
print("score_cols:", score_cols)

X_train = df_train[df_train["flag"] == "train"][col_var]
y_train = df_train[df_train["flag"] == "train"][target]
X_valid = df_train[df_train["flag"] == "valid"][col_var]
y_valid = df_train[df_train["flag"] == "valid"][target]
X_oot = df_oot[col_var]
y_oot = df_oot[target]

print(X_train.shape, X_valid.shape, X_oot.shape)

In [ ]:
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "is_unbalance": True,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "nthread": 7,
    "verbosity": -1,
    "n_estimators": 100,
    "learning_rate": 0.01,
    "num_leaves": 4,
    "lambda_l1": 30,
    "lambda_l2": 80,
    "min_data_in_leaf": 1000
}

train_set = lgb.Dataset(X_train, y_train)
gbm = lgb.train(params, train_set)

y_train_pred = gbm.predict(X_train)
y_valid_pred = gbm.predict(X_valid)
y_oot_pred = gbm.predict(X_oot)

result = pd.concat([
    auc_gini_ks(y_train_pred, y_train, "train"),
    auc_gini_ks(y_valid_pred, y_valid, "valid"),
    auc_gini_ks(y_oot_pred, y_oot, "oot")
])

display(result)

In [ ]:
save_tag = f"LGBM_含评分_4{version_tag}_{customer_type}_{target_type}"

joblib.dump(gbm, OUTPUT_DIR / f"5.model-{save_tag}.pkl")
result.to_csv(OUTPUT_DIR / f"5.模型效果_{save_tag}.csv", index=False)

fea_imp = pd.DataFrame({
    "feature_name": gbm.feature_name(),
    "feature_importance": gbm.feature_importance(importance_type="gain")
}).sort_values("feature_importance", ascending=False)

fea_imp.to_csv(OUTPUT_DIR / f"5.特征重要性_{save_tag}.csv", index=False)
display(fea_imp.head(20))

In [ ]:
A, B = solve_scorecard_params(300, 1000, y_train_pred)
print("A, B:", A, B)

score_show_lst = list(range(300, 1001, 50))

for name, y_true, pred in [
    ("train", y_train, y_train_pred),
    ("valid", y_valid, y_valid_pred),
    ("oot", y_oot, y_oot_pred)
]:
    dfp = pd.DataFrame({"y": y_true, "pred": pred})
    prob = np.clip(dfp["pred"], 1e-6, 1 - 1e-6)
    dfp["score"] = np.clip(A - B * np.log(prob / (1 - prob)), 300, 1000).round(0).astype(int)

    dt10 = model_dictionary_score(dfp, var="score", bins=10, flagy="y", risk_direction="low")
    dt50 = model_dictionary_score(dfp, var="score", bins=score_show_lst, flagy="y", risk_direction="low")

    dt10.to_csv(OUTPUT_DIR / f"5.{name}_决策表_等频_LGBM_含评分_4{version_tag}_{customer_type}_{target_type}.csv", index=False)
    dt50.to_csv(OUTPUT_DIR / f"5.{name}_决策表_等距_LGBM_含评分_4{version_tag}_{customer_type}_{target_type}.csv", index=False)